In [1]:
import json
import os
import sys
from dataclasses import dataclass, field
from pathlib import Path
from typing import Literal

import optuna
import wandb
from dotenv import load_dotenv

sys.path.append(os.path.abspath("../.."))

import src.utils.run_optuna as op
from src.utils.optuna_objective import create_objective
from src.utils.telegram import send_message

In [3]:
# === Configuration === (you edit here) ===
load_dotenv(dotenv_path="../../.env")


@dataclass
class Config:
    # Data / CV
    model_name: str = "lgbm"
    data_id: str = "057"
    n_folds: int = 5
    seed: int = 42
    fold_idx: int = 0

    # Optuna
    n_trials: int = 1
    direction: str = "maximize"
    sampler: str = "tpe"  # tpe / random
    pruner: str = "median"  # median / none

    # Initial params
    use_initial: Literal["never", "manual"] = "manual"
    initial_param_sources: list[tuple[str, int]] = field(default_factory=list)   # (study_name, n_trial) 例: ("xgb-001", 1)

    # Storage
    storage: str = "sqlite:////home/hanse/kaggle/binary-bank/artifacts/optuna/optuna.db"

    # Option
    opts: dict = field(default_factory=dict)


cfg = Config()
cfg.initial_param_sources = [("lgbm-057", 16)]

# W&B
wandb_project = os.environ.get("COMPETITION_NAME")
wandb.login(key=os.environ.get("WANDB_API_KEY"))

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: Appending key for api.wandb.ai to your netrc file: /home/hanse/.netrc


True

In [4]:
# === Build & Run (frozen) ===
# --- helper: initial params loader ---
def load_initial_params(sources: list[tuple[str, int]]) -> list[dict]:
    loaded = []
    for study, n_trial in sources:
        path = Path(f"../../artifacts/optuna/{study}/trl{n_trial}.json")
        with path.open("r") as f:
            params = json.load(f)["params"]
            params["learning_rate"] = 0.01
        loaded.append(params)
    return loaded


# sampler / pruner factory
def build_sampler(name, seed):
    if name == "tpe":
        return optuna.samplers.TPESampler(n_startup_trials=15, seed=seed)
    elif name == "random":
        return optuna.samplers.RandomSampler(seed=seed)
    else:
        raise ValueError(f"unknown sampler: {name}")


def build_pruner(name):
    if name == "median":
        return optuna.pruners.MedianPruner(n_startup_trials=10, n_warmup_steps=1000)
    elif name == "none":
        return optuna.pruners.NopPruner()
    else:
        raise ValueError(f"unknown pruner: {name}")


objective = create_objective(
    cfg.model_name,
    cfg.data_id,
    seed=cfg.seed,
    n_folds=cfg.n_folds,
    fold_idx=cfg.fold_idx,
    wandb_project=wandb_project,
    study_name=f"{cfg.model_name}-{cfg.data_id}",
    opts=cfg.opts
)

sampler = build_sampler(cfg.sampler, cfg.seed)
pruner = build_pruner(cfg.pruner)

initial_params = None
if cfg.use_initial == "manual":
    initial_params = load_initial_params(cfg.initial_param_sources)

op.run_optuna_search(
    objective,
    n_trials=cfg.n_trials,
    direction=cfg.direction,
    study_name=f"{cfg.model_name}-{cfg.data_id}",
    storage=cfg.storage,
    sampler=sampler,
    pruner=pruner,
    initial_params=initial_params
)

[I 2025-10-06 18:31:41,931] Using an existing study with name 'lgbm-057' instead of creating a new one.


[initial] 1 trial(s) will be enqueued:


  0%|          | 0/1 [00:00<?, ?it/s]

/home/hanse/miniconda3/envs/rapids-25.08/lib/python3.13/site-packages/optuna/trial/_trial.py:650: UserWarning: Fixed parameter 'learning_rate' with value 0.01 is out of range for distribution FloatDistribution(high=0.02, log=False, low=0.02, step=None).
  warnings.warn(


Fold Col: 5fold-s42
Free CPU Mem: 13.88 GB
Free GPU Mem: 6.83 GB
Training until validation scores don't improve for 1000 rounds
[100]	train's auc: 0.974274	valid's auc: 0.972672
[200]	train's auc: 0.976169	valid's auc: 0.973814
[300]	train's auc: 0.978012	valid's auc: 0.974713
[400]	train's auc: 0.979751	valid's auc: 0.975387
[500]	train's auc: 0.981379	valid's auc: 0.975877
[600]	train's auc: 0.982922	valid's auc: 0.97619
[700]	train's auc: 0.984391	valid's auc: 0.976422
[800]	train's auc: 0.985785	valid's auc: 0.976607
[900]	train's auc: 0.987084	valid's auc: 0.976735
[1000]	train's auc: 0.988325	valid's auc: 0.976834
[1100]	train's auc: 0.989424	valid's auc: 0.976922
[1200]	train's auc: 0.99045	valid's auc: 0.976974
[1300]	train's auc: 0.991378	valid's auc: 0.977024
[1400]	train's auc: 0.992232	valid's auc: 0.977063
[1500]	train's auc: 0.992988	valid's auc: 0.977083
[1600]	train's auc: 0.993684	valid's auc: 0.97711
[1700]	train's auc: 0.99431	valid's auc: 0.977138
[1800]	train's auc

iter_f1,▁▁▁▁▁▂▂▂▂▂▂▂▃▃▃▃▄▄▄▅▅▅▅▅▅▅▅▆▆▆▆▆▆▆▆▆▇▇▇█
train/f1/auc,▁▂▂▂▂▃▄▄▅▅▅▆▆▆▆▆▆▇▇▇▇▇▇▇▇▇▇▇████████████
valid/f1/auc,▁▄▅▆▇▇▇▇████████████████████████████████
auc_f1,0.9772
iter_f1,3902
runtime_f1,13.20993
train/f1/auc,0.99951
valid/f1/auc,0.97716


[I 2025-10-06 18:46:22,422] Trial 21 finished with value: 0.9772024401995091 and parameters: {'learning_rate': 0.01, 'num_leaves': 970, 'min_child_samples': 362, 'min_split_gain': 0.00045914418634831735, 'feature_fraction': 0.37121824466699876, 'bagging_fraction': 0.9049166225938291, 'bagging_freq': 8, 'lambda_l1': 8.991609072217798, 'lambda_l2': 0.13082234598927622}. Best is trial 21 with value: 0.9772024401995091.
✅ Message sent.
